# AURA Frontier Validation Notebook

This notebook is a **cloud/remote validation harness**, not AURA production execution. It downloads a pinned Hugging Face snapshot in the active runtime, inspects it, optionally executes it through AirLLM or Transformers, records only measurements returned by the process, and writes JSON/JSONL/CSV/Markdown artifacts.

**Truth rules:** `REAL` means a live process measured the value; `PlannerEstimated` is reserved for AURA planner output and is never substituted into measured fields; `Simulated` is not used as evidence; `Remote/Cloud execution` is always stated separately from local execution. A model that cannot download, initialize, or generate is `NOT_EXECUTED`/`FAILED` with the exception and environment recorded.

The notebook intentionally does not download frontier checkpoints on the developer machine. Run it in Google Colab, Kaggle, or a GPU VM with sufficient disk. Large checkpoints may require more disk than the VM provides; in that case the correct result is `NOT_EXECUTED`.

In [ ]:
# Cloud environment setup. Run in Colab/Kaggle/VM, not on the low-end development machine.
%pip -q install -U airllm transformers accelerate safetensors huggingface_hub psutil pynvml nbformat


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform
ROOT = Path.cwd()
RUNNER = ROOT / "notebooks" / "frontier_validation_runner.py"
if not RUNNER.exists():
    # When opened directly from the notebook directory
    RUNNER = Path("frontier_validation_runner.py")
assert RUNNER.exists(), f"Runner not found: {RUNNER}"
print("runner:", RUNNER.resolve())
print("python:", sys.version)
print("platform:", platform.platform())
try:
    print(subprocess.check_output(["nvidia-smi"], text=True)[:2000])
except Exception as e:
    print("No nvidia-smi; GPU execution may be unavailable:", repr(e))


## Select a model and immutable revision

Use a real Hugging Face repository and preferably a commit SHA. Names such as GLM-5.2 or Kimi K3 must not be guessed: if the repository ID is unavailable or gated, the notebook records that failure. Set `HF_TOKEN` in the cloud runtime for gated repositories; do not commit the token.

In [ ]:
# Change these values before running. The default is a small public smoke model so the notebook is executable
# without implicitly attempting a multi-hundred-GB download. Replace with a real frontier repo and pinned SHA.
MODEL_REPO = os.getenv("AURA_MODEL_REPO", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_REVISION = os.getenv("AURA_MODEL_REVISION") or None
HF_TOKEN = os.getenv("HF_TOKEN")
BACKEND = os.getenv("AURA_BACKEND", "airllm")  # airllm or transformers
CACHE_DIR = os.getenv("HF_HOME") or None
ARTIFACT_DIR = Path(os.getenv("AURA_ARTIFACT_DIR", "artifacts/frontier"))
MAX_NEW_TOKENS = int(os.getenv("AURA_MAX_NEW_TOKENS", "32"))
PROMPT_COUNT = int(os.getenv("AURA_PROMPT_COUNT", "50"))
print({"repo":MODEL_REPO,"revision":MODEL_REVISION,"backend":BACKEND,"artifacts":str(ARTIFACT_DIR),"prompt_count":PROMPT_COUNT})


## AURA integration boundary

The current AURA repository is Rust-first. This notebook uses AURA’s existing model/planner concepts where available, but does not pretend that the current Rust CLI is a Hugging Face/AirLLM execution backend. If a built AURA CLI is present, run it for inspection/planning and preserve its output as `PlannerEstimated` or `ARCHITECTURE ONLY`; the live generation measurements below come from the selected Python backend and are labeled separately.

In [ ]:
# Optional AURA planner/inspector call. Missing binary is an explicit capability gap, not a success.
AURA_BIN = os.getenv("AURA_BIN", "aura")
try:
    p = subprocess.run([AURA_BIN, "frontier", "inspect", MODEL_REPO], capture_output=True, text=True, timeout=120)
    aura_planner = {"status":"AVAILABLE" if p.returncode == 0 else "FAILED", "execution_class":"PlannerEstimated", "returncode":p.returncode, "stdout":p.stdout, "stderr":p.stderr}
except Exception as e:
    aura_planner = {"status":"NOT_AVAILABLE", "execution_class":"ARCHITECTURE ONLY", "error":f"{type(e).__name__}: {e}"}
print(json.dumps(aura_planner, indent=2)[:5000])


In [ ]:
# Model inspection performs a real Hugging Face snapshot download/cache resolution and reports disk/config facts.
from importlib.util import spec_from_file_location, module_from_spec
spec = spec_from_file_location("aura_frontier_runner", RUNNER)
runner = module_from_spec(spec); spec.loader.exec_module(runner)
inspection = runner.inspect_model(MODEL_REPO, MODEL_REVISION, HF_TOKEN, CACHE_DIR)
print(json.dumps(inspection, indent=2, default=str)[:12000])


## Execute real generation

The next cell attempts all 50 prompts by default. It measures per-process load time, RSS, NVIDIA memory snapshots, disk footprint, token counts, total generation time, and decode tok/s. TTFT is reported only when the backend exposes a real first-token timestamp; this runner leaves it `null` rather than inferring it from total time. AirLLM hook counters are observational instrumentation around its actual layer/expert loading hooks. A missing counter is not evidence that no transfer occurred.

In [ ]:
result = runner.run_model(MODEL_REPO, MODEL_REVISION, BACKEND, runner.PROMPTS[:PROMPT_COUNT], MAX_NEW_TOKENS, HF_TOKEN, CACHE_DIR, True)
print(json.dumps(result, indent=2, default=str)[:20000])


In [ ]:
# Persist all artifacts. Existing files are overwritten for this run directory.
runner.write_artifacts([result], ARTIFACT_DIR)
print("Wrote:", *sorted(str(p) for p in ARTIFACT_DIR.glob("*")), sep="\n")


## Interpretation

`EXECUTED` means at least one prompt produced at least one token in this runtime. It does not mean the model ran locally: consult `execution_scope` and `hardware_after`. `LOADED` without generation is not model support. `NOT_EXECUTED` means no generation evidence exists. `PARTIAL_OOM` and `PARTIAL_FAILURE` retain successful and failed prompt records. Do not replace nulls with estimates. Commit the artifacts together with the exact model revision, hardware profile, command/environment, and notebook revision.